## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## Run

In [3]:
# ✅ Agents developed under `agentic/agents`:
#   - classifier.py  : Deterministic ticket classification (category + urgency)
#   - resolver.py    : Knowledge-base retrieval + tool-assisted resolution
#   - escalation.py  : Builds escalation packages for unresolvable/critical issues
#   - memory_agent.py: Long-term memory - retrieves/stores customer interaction history
#
# ✅ Tools developed under `agentic/tools`:
#   - account_tools.py : lookup_user_record, get_subscription_details
#   - ticket_tools.py  : update_ticket_status, log_ticket_message, get_ticket_history
#   - knowledge_tools.py: search_knowledge_base (semantic retrieval from articles)
#
# ✅ Workflow orchestration in `agentic/workflow.py`:
#   - Supervisor pattern using LangGraph StateGraph
#   - Entry: memory → supervisor → classifier → supervisor → resolver/escalation → END
#   - MemorySaver checkpointer for session & thread-level state persistence
#   - Conditional routing based on classification, urgency, and resolution confidence

# Verify agent modules are importable
from agentic.agents.classifier import classify_ticket
from agentic.agents.resolver import resolve_ticket
from agentic.agents.escalation import build_escalation
from agentic.agents.memory_agent import resolve_customer_context
from agentic.tools.account_tools import lookup_user_record
from agentic.tools.ticket_tools import update_ticket_status, log_ticket_message, get_ticket_history
from agentic.tools.knowledge_tools import search_knowledge_base

print("✅ All agents and tools imported successfully")
print(f"   Agents: classifier, resolver, escalation, memory_agent")
print(f"   Tools: lookup_user_record, update_ticket_status, log_ticket_message, get_ticket_history, search_knowledge_base")

✅ All agents and tools imported successfully
   Agents: classifier, resolver, escalation, memory_agent
   Tools: lookup_user_record, update_ticket_status, log_ticket_message, get_ticket_history, search_knowledge_base


In [4]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

## Architecture Overview

The multi-agent system uses a **Supervisor pattern** orchestrated with LangGraph:

```
User Message → [Memory Agent] → [Supervisor] → [Classifier] → [Supervisor] → [Resolver/Escalation] → Response
```

| Agent | Role | Responsibility |
|-------|------|----------------|
| **Memory** | Context retrieval | Matches users, retrieves past interactions, stores preferences |
| **Classifier** | Routing | Categorizes tickets (billing/technical/account/etc.) and assigns urgency |
| **Resolver** | Resolution | Searches knowledge base, invokes tools, generates responses |
| **Escalation** | Fallback | Creates escalation packages for critical/unresolvable issues |
| **Supervisor** | Orchestrator | Routes between agents based on state (classification, confidence, urgency) |

**State persistence**: `MemorySaver` checkpointer enables session inspection via `thread_id`.

In [5]:
# Demonstrate the graph structure
print("Workflow graph nodes:", list(orchestrator.get_graph().nodes.keys()))
print("Entry point: memory → supervisor (conditional routing)")
print()

# Quick end-to-end test without interactive chat
from langchain_core.messages import HumanMessage

test_result = orchestrator.invoke(
    {"messages": [HumanMessage(content="I can't log in to my CultPass account")]},
    config={"configurable": {"thread_id": "test-demo-1"}}
)

print("--- End-to-End Demo ---")
print(f"Input: 'I can't log in to my CultPass account'")
print(f"Classification: {test_result.get('classification')}")
print(f"Urgency: {test_result.get('urgency')}")
print(f"Resolution confidence: {test_result.get('resolution_confidence')}")
print(f"Tools used: {test_result.get('tool_usage', [])}")
print(f"Response: {test_result['messages'][-1].content[:200]}...")

Workflow graph nodes: ['__start__', 'supervisor', 'classifier', 'resolver', 'escalation', 'memory', '__end__']
Entry point: memory → supervisor (conditional routing)

--- End-to-End Demo ---
Input: 'I can't log in to my CultPass account'
Classification: account
Urgency: high
Resolution confidence: HIGH
Tools used: ['search_knowledge_base']
Response: I found relevant guidance in What's Included in a CultPass Subscription, How to Handle Blocked Accounts.

Each user is entitled to 4 cultural experiences per month, which may include:
- Art exhibition...


In [6]:
chat_interface(orchestrator, "1")

User: 
Assistant: (Please type a message or 'quit' to exit)
User: 
Assistant: (Please type a message or 'quit' to exit)
User: hi
Assistant: I understand your concern, and I want to make sure you get the best possible help.

I've escalated your case to our specialized support team with P4 priority.

Summary: General issue: hi

Your case has been escalated with P4 priority. A human support specialist will review your general issue and follow up shortly.
User: hihihihihih
Assistant: I understand your concern, and I want to make sure you get the best possible help.

I've escalated your case to our specialized support team with P4 priority.

Summary: General issue: hihihihihih

Your case has been escalated with P4 priority. A human support specialist will review your general issue and follow up shortly.
User: 
Assistant: (Please type a message or 'quit' to exit)
User: 
Assistant: (Please type a message or 'quit' to exit)
User: quit
Assistant: Goodbye!


In [8]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='[Memory] No prior customer history found.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content='[Classifier] Category: general, Urgency: low', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content='I could not find a reliable knowledge base article for this request. I recommend escalating it to a human support specialist so they can investigate further.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content="I understand your concern, and I want to make sure you get the best possible help.\n\nI've escalated your case to our specialized support team with P4 priority.\n\nSummary: General issue: hi\n\nYour case has been escalated with P4 priority. A human support specialist will review your general issue and follow up shortly.", additional_kwargs={}, r